<a href="https://colab.research.google.com/github/mamunh9/Autonomous-Systems-Lab/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Manual Spam Filter vs Machine Learning Spam Filter

## Learning objectives
After working through this notebook, you should be able to:
- explain the difference between classical rule-based programming and machine learning,
- implement a simple manual spam filter,
- understand why manual rules become difficult to maintain,
- create a small text dataset,
- transform text into numerical features,
- train a simple scikit-learn classifier,
- use `.fit(...)`, `.predict(...)`, and `.score(...)`.

## Context
The introductory lecture discussed the example of a spam filter:

- **Classical programming**: write many explicit rules manually.
- **Machine learning**: learn patterns from labeled examples.

This notebook follows exactly this idea step by step.

## References
1. Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, O'Reilly, introductory chapters.  
2. scikit-learn User Guide: https://scikit-learn.org/stable/user_guide.html  
3. scikit-learn Text Feature Extraction: https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction  
4. scikit-learn Naive Bayes: https://scikit-learn.org/stable/modules/naive_bayes.html  
5. Jake VanderPlas, *Python Data Science Handbook*, O'Reilly, Chapter 5.

# Scientific Background: From Manual Rules to Machine Learning

This notebook introduces machine learning from a scientific and engineering perspective.

We consider a **binary classification problem**.  
Each message \(x_i\) must be assigned to one of two classes:

\[$
y_i \in \{0,1\}
$\]

where:

- \(0\): not spam
- \(1\): spam

The goal is to define or learn a function:

\[$
f(x) \rightarrow y
$\]

that maps an input message to a class label.

In the classical programming approach, this function is explicitly written by the programmer using rules.  
In the machine learning approach, the function is learned from labeled examples.

This comparison is important because it reflects one of the central motivations of machine learning:
instead of maintaining long rule lists manually, we use data to infer patterns.

## Part 1: The problem

We want to classify short messages as:

- `1`: spam
- `0`: not spam

Example:

```text
"win money now" -> spam
"meeting tomorrow" -> not spam
```

This is a **classification task** because the output is a discrete class.

## Scientific view: rule-based classification

The manual spam filter is a **symbolic rule-based system**.

The knowledge is encoded explicitly in conditions such as:

```python
if "free" in message:
    return 1
```

This means that the classification function is manually designed.

### Advantages
- transparent and easy to inspect,
- no training data required,
- predictable behavior for known cases.

### Limitations
- difficult to scale,
- fragile when language changes,
- many exceptions require new rules,
- maintenance effort increases over time.

This corresponds to the introductory lecture idea: traditional programming may require long and complex rule lists.

## Part 2: Manual spam filter

In classical programming, the programmer writes explicit rules.

Run the next cell.

In [1]:
def manual_spam_filter(message):
    message = message.lower()

    if "win" in message:
        return 1
    if "free" in message:
        return 1

    return 0

In [2]:
messages = [
    "win money now",
    "hello friend",
    "free lottery",
    "meeting tomorrow"
]

for message in messages:
    prediction = manual_spam_filter(message)
    print(message, "->", prediction)

win money now -> 1
hello friend -> 0
free lottery -> 1
meeting tomorrow -> 0


### Task 4A.1
Test the manual spam filter with the following messages:

- `"free prize"`
- `"project meeting"`
- `"winner announcement"`

### Reflection
Which message is incorrectly classified or not detected well?

In [3]:
# TODO: test additional messages
messages = [
    "free prize",
    "project meeting",
    "winner announcement"
]

for message in messages:
    prediction = manual_spam_filter(message)
    print(message, "->", prediction)


free prize -> 1
project meeting -> 0
winner announcement -> 1


## Scientific view: manual update

When the rule-based system fails, the update process is:

\[$
\text{observed error} \rightarrow \text{human analysis} \rightarrow \text{new rule}
$\]

This is a manual engineering process.

For a small example, this is manageable.  
For realistic spam filtering, it becomes problematic because spam messages continuously change:
- new words,
- spelling variations,
- abbreviations,
- misleading phrases,
- different languages.

Therefore, the manual approach is useful for understanding but limited as a scalable solution.

## Part 3: Manual update of the spam filter

Assume we observe a new spam message:

```text
"winner announcement"
```

Our current filter may not classify it as spam because it only checks for `"win"` and `"free"`.

In a manual system, we must update the rules ourselves.

### Task 4A.2
Create a new function called `manual_spam_filter_updated`.

Add a new rule for the word `"winner"`.

Then test both:
- `"winner announcement"`
- `"project meeting"`

In [10]:
# TODO: implement updated manual spam filter
def manual_spam_filter_updated(message):
    message = message.lower()

    if "win" in message:
        return 1
    if "free" in message:
        return 1
    if "winner" in message:
        return 1

    return 0


messages = [
    "winner announcement",
    "project meeting"
]

for message in messages:
    prediction = manual_spam_filter_updated(message)
    print(message, "->", prediction)


winner announcement -> 1
project meeting -> 0


### Reflection
Manual update means:
- observe failure,
- inspect the case,
- write a new rule,
- test again.

Why can this become difficult for a real spam filter?

## Scientific view: supervised machine learning

In supervised machine learning, we use a labeled dataset:

\[
D = \{(x_1,y_1), (x_2,y_2), \ldots, (x_n,y_n)\}
\]

where:

- \(x_i\) is one input example, here a text message,
- \(y_i\) is the known target label,
- \(n\) is the number of examples.

The learning algorithm estimates a model:

\[
\hat{f}(x) \approx y
\]

The model should not only reproduce known examples but also generalize to new, unseen messages.

The key shift is:

| Classical programming | Machine learning |
|---|---|
| programmer writes rules | model learns patterns |
| update means changing code | update means adding data and retraining |
| knowledge is in the source code | knowledge is in the trained model |

## Part 4: A small labeled dataset

A machine learning approach needs examples.

We create:
- `texts`: input messages,
- `labels`: desired outputs.

The labels are:
- `1`: spam
- `0`: not spam

In [11]:
texts = [
    "win money now",
    "free lottery prize",
    "winner claim your prize",
    "cheap offer now",
    "hello friend",
    "meeting tomorrow",
    "project discussion",
    "lunch at noon"
]

labels = [1, 1, 1, 1, 0, 0, 0, 0]

print("Number of texts:", len(texts))
print("Number of labels:", len(labels))

Number of texts: 8
Number of labels: 8


### Task 4A.3
Print each text together with its label.

Hint:
```python
for text, label in zip(texts, labels):
    ...
```

In [12]:
# TODO: print text and label pairs
for text, label in zip(texts, labels):
    print(text, "->", label)

win money now -> 1
free lottery prize -> 1
winner claim your prize -> 1
cheap offer now -> 1
hello friend -> 0
meeting tomorrow -> 0
project discussion -> 0
lunch at noon -> 0


## Scientific view: feature extraction

Most machine learning algorithms cannot directly process raw text.
Therefore, text must be transformed into numerical features.

This step is called **feature extraction**.

In this notebook, we use a simple **bag-of-words** representation:

1. collect all words in the training texts,
2. create a vocabulary,
3. represent each message by word counts.

Example:

```text
"free money" -> [1, 0, 1, 0, ...]
```

Mathematically, each message becomes a feature vector:

\[
x_i = (x_{i1}, x_{i2}, \ldots, x_{id})
\]

where \(d\) is the number of vocabulary words.

Important simplification:
- word order is ignored,
- grammar is ignored,
- only word occurrence counts are used.

This representation is simple, but it is a good starting point for understanding text classification.

## Part 5: Text must become numbers

Machine learning models cannot directly work with raw text.

We need to convert text into numerical features.

A simple method is **bag-of-words**:
- collect all words,
- count how often each word appears in each message.

In scikit-learn, this can be done with `CountVectorizer`.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

print("Feature matrix shape:", X.shape)
print("Vocabulary:")
print(vectorizer.get_feature_names_out())

Feature matrix shape: (8, 20)
Vocabulary:
['at' 'cheap' 'claim' 'discussion' 'free' 'friend' 'hello' 'lottery'
 'lunch' 'meeting' 'money' 'noon' 'now' 'offer' 'prize' 'project'
 'tomorrow' 'win' 'winner' 'your']


### Interpretation

`X.shape` has the form:

```python
(number_of_messages, number_of_words_in_vocabulary)
```

### Task 4A.4
Answer:
1. How many messages are in the dataset?
2. How many different words are in the vocabulary?
3. Why is this transformation necessary?

8

20

To convert text into numerical features so machine learning models can process it.

## Part 6: Inspect the numerical representation

The result `X` is stored in a sparse matrix format.
For learning purposes, we convert it into a dense table.

Run the next cell.

In [14]:
import pandas as pd

X_table = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out()
)

X_table

,at,cheap,claim,discussion,free,friend,hello,lottery,lunch,meeting,money,noon,now,offer,prize,project,tomorrow,win,winner,your
0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0
1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1
3,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0
4,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0
6,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
7,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0


### Task 4A.5
Find the row corresponding to `"win money now"`.

Which words have count `1` in this row?

## Scientific view: model selection

We use **Multinomial Naive Bayes** as the first text classification model.

It is a common baseline for text classification because:
- it works naturally with count data,
- it is computationally simple,
- it often performs surprisingly well for simple text tasks,
- it is suitable for teaching the basic workflow.

The model learns statistical relationships between words and classes.

For example, it may learn that words like:
- "free",
- "winner",
- "lottery"

occur more often in spam messages, while words like:
- "meeting",
- "project",
- "lunch"

occur more often in non-spam messages.

## Part 7: Train a machine learning classifier

We now train a simple classifier.

We use **Multinomial Naive Bayes**, a common baseline model for text classification.

In [15]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X, labels)

MultinomialNB()

### Reflection
What are the inputs to `.fit(...)` here?

- What is `X`?
- What are `labels`?

## Scientific view: training and prediction

Training is performed with:

```python
model.fit(X, labels)
```

Here:
- \(X\) is the feature matrix,
- `labels` is the target vector.

Prediction is performed with:

```python
model.predict(X_new)
```

A new text must first be transformed using the same vectorizer:

```python
X_new = vectorizer.transform(new_messages)
```

This is essential: training and prediction must use the same feature representation.

The complete prediction pipeline is:

\[
\text{new text} \rightarrow \text{feature vector} \rightarrow \text{trained model} \rightarrow \text{predicted class}
\]

## Part 8: Predict new messages

To predict a new text, we must apply the same text transformation.

Important:
- training texts: `vectorizer.fit_transform(...)`
- new texts: `vectorizer.transform(...)`

Run the next cell.

In [16]:
new_messages = [
    "free money",
    "project meeting",
    "winner prize"
]

X_new = vectorizer.transform(new_messages)
predictions = model.predict(X_new)

for message, prediction in zip(new_messages, predictions):
    print(message, "->", prediction)

free money -> 1
project meeting -> 0
winner prize -> 1


### Task 4A.6
Add at least three own messages and predict their class.

Use:
```python
my_messages = [...]
X_my = vectorizer.transform(my_messages)
model.predict(X_my)
```

In [17]:
# TODO: predict own messages
my_messages=[
    "withdrow now",
    "what`s up?",
    "claim your free reward"

]
X_my= vectorizer.transform(my_messages)
predictions = model.predict(X_my)

for message, prediction in zip(my_messages, predictions):
    print(message, "->", prediction)



withdrow now -> 1
what`s up? -> 0
claim your free reward -> 1


## Scientific view: comparison and update

The comparison of manual and ML filters shows two different engineering strategies.

### Manual update
\[
\text{new case} \rightarrow \text{new hand-written rule}
\]

### Machine learning update
\[
\text{new labeled examples} \rightarrow \text{retraining} \rightarrow \text{updated model}
\]

However, machine learning is not automatic magic.

The quality of the model depends on:
- quality of the dataset,
- number of examples,
- representative examples,
- suitable feature extraction,
- meaningful evaluation.

A small toy dataset is useful for learning the workflow, but real-world systems require careful validation.

## Part 9: Compare manual rules and machine learning

We now compare both approaches on the same messages.

In [18]:
test_messages = [
    "free money",
    "winner prize",
    "project meeting",
    "cheap lottery",
    "lunch tomorrow"
]

print("Manual filter:")
for message in test_messages:
    print(message, "->", manual_spam_filter(message))

print()
print("Machine learning filter:")
X_test_messages = vectorizer.transform(test_messages)
ml_predictions = model.predict(X_test_messages)

for message, prediction in zip(test_messages, ml_predictions):
    print(message, "->", prediction)

Manual filter:
free money -> 1
winner prize -> 1
project meeting -> 0
cheap lottery -> 0
lunch tomorrow -> 0

Machine learning filter:
free money -> 1
winner prize -> 1
project meeting -> 0
cheap lottery -> 1
lunch tomorrow -> 0


### Task 4A.7
Discuss:

1. Which approach is easier to understand?
2. Which approach is easier to update when many new spam examples appear?
3. What is the role of data in the machine learning approach?

## Part 10: Add more training data

A machine learning system can be improved by adding more examples.

### Task 4A.8
Extend the dataset with at least:
- two new spam messages,
- two new non-spam messages.

Then:
1. recreate the vectorizer,
2. recreate `X`,
3. train a new model,
4. test the new model.

This is the machine learning version of an update.

In [19]:
# TODO: extend dataset and retrain model
texts = [
    "win money now",
    "free lottery prize",
    "winner claim your prize",
    "cheap offer now",
    "hello friend",
    "meeting tomorrow",
    "project discussion",
    "lunch at noon",
    "withdrow now",
    "limited time offer",
    "see you later",
    "dinner at my place"
]

labels = [1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0]

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(texts)

from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X, labels)

test_messages = [
    "free prize now",
    "team meeting today",
    "win a free ticket",
    "let's have lunch"
]

X_test = vectorizer.transform(test_messages)
predictions = model.predict(X_test)

for message, prediction in zip(test_messages, predictions):
    print(message, "->", prediction)


free prize now -> 1
team meeting today -> 0
win a free ticket -> 1
let's have lunch -> 0


## Summary

You compared two approaches:

### Classical programming
- rules are written manually,
- updates require new manual rules,
- understandable but difficult to scale.

### Machine learning
- rules/patterns are learned from examples,
- updates can be done by adding data and retraining,
- requires data preparation and evaluation.

This notebook connects directly to the seminar topic:
**Machine Learning vs. Traditional Programming**.

## Main learning outcome

Machine learning does not remove the need for human thinking.

It changes the engineering task:

Instead of manually writing many rules, we must:
1. define the learning problem,
2. collect and label data,
3. choose a representation,
4. select a model,
5. train the model,
6. evaluate the result,
7. update the system using new data.

This notebook is therefore a conceptual bridge between the introductory lecture and later seminar topics such as:
- classification,
- feature engineering,
- model evaluation,
- text classification,
- data quality,
- generalization.